#### Using the same idea as the interpolation of resident numbers

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [6]:
# get subzone data with their corresponding areas in km^2
def get_subzone_data(year: int):
    """
    Args
    ------
    year: int
        year of the subzones you want (2019, 2014, 2008)

    Returns
    ------
    Subzone dataframe containing the subzone names, planning area names and areas of points of interest.
    """

    year = str(year)

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_boundary_{year}.gpkg")
    subzones_gpd = gpd.read_file(subzones_filepath)
    subzones_gpd = subzones_gpd.map(lambda s: s.lower() if isinstance(s, str) else s)
    # lower() the column names of subzones_gpd
    subzones_gpd.columns = subzones_gpd.columns.str.lower()    

    # there is no landuse data found for 2008 masterplan, 
    # so not able to merge with the subzone_classification file obtained from extracting_landuse_type.ipynb
    if year == "2008":
        return subzones_gpd

    subzones_filepath = Path(BASE_DATASET_PATH / f"singapore_data/data_gov/masterplan_{year}/subzone_classifications_{year}.csv")
    subzones_csv = pd.read_csv(subzones_filepath)
    subzones_csv = subzones_csv.map(lambda s: s.lower() if isinstance(s, str) else s)

    subzones_with_geom = subzones_csv.merge(
        subzones_gpd[["subzone_n", "geometry"]],
        on = "subzone_n",
        how = "left"
    )

    missing_count = subzones_with_geom[subzones_with_geom.columns[-1]].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} subzones did not find a match in the CSV.")

    return subzones_with_geom

In [4]:
def get_ethnicity_data(year: int):

    year = str(year)

    ethnicity_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ethnicity_combined.xlsx")
    ethnicity_df = pd.read_excel(ethnicity_filepath, sheet_name = f"{year}")

    ethnicity_df = ethnicity_df.rename(columns = {
        "subzone": "subzone_n",
        "planning_area": "pln_area_n"
    })

    ethnicity_df["pln_area_n"] = ethnicity_df["pln_area_n"].ffill()

    ## the subzones for punggol is named differently in the masterplan for 2010 and 2015, 
    # instead of eg: subzone 2, it is sz2, so renaming it
    if year == "2015" or year == "2010":
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 2") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz2"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 3") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz3"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 4") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz4"
        ethnicity_df.loc[
            (ethnicity_df["subzone_n"] == "subzone 5") &
            (ethnicity_df["pln_area_n"] == "punggol"), 
            "subzone_n"
        ] = "sz5"

    return ethnicity_df

In [ ]:
def convert_to_geodataframe(df):
    if not isinstance(df, gpd.GeoDataFrame):
        df = gpd.GeoDataFrame(df, geometry='geometry')

    # ensure the original data has the correct starting CRS (WGS84)
    # (Only needed if it isn't already set; usually .gpkg files have this metadata)
    if df.crs is None:
        df.set_crs(epsg=4326, inplace=True)

    # transform the entire GeoDataFrame to SVY21 (Meters)
    df_meters = df.to_crs(epsg=3414)

    return df_meters

In [13]:
subzone_df = get_subzone_data(2008)
subzone_df = convert_to_geodataframe(subzone_df)
subzone_df['area_sqm'] = subzone_df.geometry.area
subzone_df['area_sqkm'] = subzone_df['area_sqm'] / 1_000_000
subzone_areas = subzone_df[['subzone_n', 'pln_area_n', 'area_sqm', 'area_sqkm', "geometry"]].copy()

ethnicity_df = get_ethnicity_data(2010)


In [9]:
ethnicity_df.columns

Index(['pln_area_n', 'subzone_n', 'total', 'male_total', 'female_total',
       'total_chinese', 'male_chinese', 'female_chinese', 'total_malays',
       'male_malays', 'female_malays', 'total_indians', 'male_indians',
       'female_indians', 'total_others', 'male_others', 'female_others'],
      dtype='object')

In [14]:
def calculate_density(ethnicity_df, subzone_areas, year:int):
    
    year = str(year)

    ethnicity_columns = ['pln_area_n', 'subzone_n',
       'male_chinese', 'female_chinese',
       'male_malays', 'female_malays',
       'male_indians', 'female_indians',
       'male_others', 'female_others']
        
    ethnicity_df = ethnicity_df[ethnicity_columns].copy()

    # standardize casing and remove hidden whitespace
    subzone_areas['subzone_n'] = subzone_areas['subzone_n'].astype(str).str.strip().str.lower()
    ethnicity_df['subzone_n'] = ethnicity_df['subzone_n'].astype(str).str.strip().str.lower()

    density_df = subzone_areas.merge(
        ethnicity_df,
        on = ["subzone_n", "pln_area_n"],
        how = "left"
    )
    
    for ethinicty_gender in ethnicity_columns[2:]:
        new_column_name = "density_" + ethinicty_gender

        density_df[f"density_{ethinicty_gender}"] =  density_df[ethinicty_gender] / density_df["area_sqkm"]

    return density_df
    

In [15]:
density_df = calculate_density(ethnicity_df, subzone_areas, 2010)
density_df

,subzone_n,pln_area_n,area_sqm,area_sqkm,geometry,male_chinese,female_chinese,male_malays,female_malays,male_indians,...,male_others,female_others,density_male_chinese,density_female_chinese,density_male_malays,density_female_malays,density_male_indians,density_female_indians,density_male_others,density_female_others
0,woodlands regional centre,woodlands,5.956521e+05,0.595652,"MULTIPOLYGON (((22629.486 46980.246, 22683.012...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,kranji,sungei kadut,3.654121e+06,3.654121,"MULTIPOLYGON (((20329.947 47234.988, 20333.366...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,turf club,sungei kadut,3.290816e+06,3.290816,"MULTIPOLYGON (((21094.367 44815.121, 21099.854...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,nee soon,yishun,2.209311e+06,2.209311,"MULTIPOLYGON (((26860.555 43913.328, 26839.719...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,yishun west,yishun,1.517705e+06,1.517705,"MULTIPOLYGON (((28228.609 45792.488, 28223.039...",20417.0,20694.0,5457.0,5408.0,3385.0,...,793.0,840.0,13452.551397,13635.063849,3595.561198,3563.275601,2230.341699,2124.919344,522.499547,553.467364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306,sungei serangoon west,sengkang,1.572812e+06,1.572812,"MULTIPOLYGON (((37009.969 40867.361, 37011.087...",21915.0,22183.0,3199.0,3269.0,2397.0,...,585.0,646.0,13933.646580,14104.042075,2033.937276,2078.443562,1524.022398,1512.577924,371.945391,410.729441
307,sungei serangoon,hougang,1.333531e+06,1.333531,"MULTIPOLYGON (((36538.391 39761.352, 36517.439...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,hougang central,hougang,2.291220e+06,2.291220,"MULTIPOLYGON (((35948.803 40264.817, 35951.927...",20669.0,20927.0,1805.0,1840.0,2031.0,...,463.0,577.0,9020.957126,9133.560877,787.789811,803.065514,886.427206,874.643093,202.075724,251.830870
309,changi west,changi,4.849019e+06,4.849019,"MULTIPOLYGON (((45347.86 41116.219, 45350.666 ...",305.0,338.0,186.0,147.0,199.0,...,22.0,22.0,62.899321,69.704821,38.358274,30.315410,41.039229,32.790138,4.537000,4.537000
